# Shift-Scan: matched seed x variant matrix (Kaggle)

Fills the gaps found in review: **many seeds, one config, one platform, a nearest baseline, a full-precision baseline, hang protection, per-run saving, resume across sessions, paired statistics.**

**Kaggle setup (once)**
1. Settings: Accelerator = GPU T4 (single GPU is used), Internet = On, Persistence = Files only.
2. Add the project as a Dataset input (the *advanced* `shift_scan` zip that has `--decay-quantizer`). The notebook finds it itself.
3. Add `HF_TOKEN` under Add-ons > Secrets (optional, faster downloads).
4. Run All. To continue in a new session, add this notebook's previous *output* as an input; finished runs are detected and skipped.

Every run is saved the moment it finishes. Nothing here reports a result that was not executed.

In [ ]:
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, threading, signal, zipfile, datetime, math
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
WORK, PROJECT = Path('/kaggle/working'), Path('/kaggle/working/shift_scan')
RESULTS, RUNS = WORK / 'results', WORK / 'runs'
for d in (RESULTS, RUNS):
    d.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('No HF_TOKEN secret (optional)')

REQUIRED_FLAG = '--decay-quantizer'
def find_project():
    cands = [Path(p).parent for p in glob.glob('/kaggle/input/**/train_qat.py', recursive=True)]
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('train_qat.py') for n in zf.namelist()):
                    out = WORK / '_unzipped' / Path(z).stem
                    zf.extractall(out)
                    cands += [Path(p).parent for p in glob.glob(str(out / '**' / 'train_qat.py'), recursive=True)]
        except zipfile.BadZipFile:
            pass
    good = [c for c in cands if REQUIRED_FLAG in (c / 'train_qat.py').read_text()]
    if not good:
        raise FileNotFoundError('No project with train_qat.py supporting ' + REQUIRED_FLAG +
                                ' found under /kaggle/input. Attach the advanced shift_scan zip as a Dataset.')
    return good[0]

if not PROJECT.exists():
    shutil.copytree(find_project(), PROJECT)
os.chdir(PROJECT)
print('Project:', PROJECT, sorted(p.name for p in PROJECT.glob('*.py')))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lm-eval>=0.4,<0.5', 'datasets>=2.18',
                'transformers>=4.40', 'safetensors>=0.4', 'pynvml', 'scipy', 'pandas', 'matplotlib'], check=False)

import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
# ---------------- CONFIG: one config for every run ----------------
PRETRAINED   = 'state-spaces/mamba2-130m'
TOKENS       = 2_000_000            # matches the real_harpq_fast2M protocol (976 steps)
SEQ_LEN, MICRO_BS, ACCUM, LR = 512, 1, 4, 1e-4
DTYPE        = 'fp16'               # T4 has no native bf16; use the same dtype for train AND eval
SEEDS        = [0, 1, 2, 3, 4]
VARIANTS_TO_RUN = ['nearest', 'harpq', 'dyadic2', 'fp_student']   # add 'ternary_expdecay' to isolate weight vs decay quantization
EVAL_TASKS   = 'lambada_openai,wikitext,piqa,arc_easy'            # fast set; use FULL_EVAL_TASKS for the final table
FULL_EVAL_TASKS = 'lambada_openai,wikitext,hellaswag,piqa,arc_easy,arc_challenge,winogrande'
SESSION_BUDGET_H = 10.5             # stop launching runs when the next one would not fit
KEEP_CKPT    = False                # checkpoints are ~2 GB with optimizer state; keep only if you need them
HANG_GRACE_S = 90                   # process still alive this long after finishing => killed (fixes the exit hang)
TRAIN_TIMEOUT_S, EVAL_TIMEOUT_S = 3 * 3600, 2 * 3600

SHIFT = ['--decay', 'shift', '--decay-group-size', '1']
BASE = ['--pretrained', PRETRAINED, '--ternary', '1', '--group-size', '128', '--activation-aware', '1',
        '--state-weight', '0.05', '--state-loss-type', 'hybrid', '--tokens', str(TOKENS),
        '--seq-len', str(SEQ_LEN), '--micro-bs', str(MICRO_BS), '--accum', str(ACCUM), '--lr', str(LR),
        '--dtype', DTYPE, '--val-seqs', '64', '--eval-every', '100', '--save-every', '250', '--log-every', '10']
VARIANTS = {   # variant flags come after BASE, so they override it (argparse: last one wins)
    'nearest':  SHIFT + ['--decay-quantizer', 'nearest'],
    'harpq':    SHIFT + ['--decay-quantizer', 'horizon', '--decay-horizon', '128', '--decay-lag-weight', 'uniform'],
    'dyadic2':  SHIFT + ['--decay-quantizer', 'dyadic', '--dyadic-terms', '2', '--decay-horizon', '128'],
    'fp_student': ['--decay', 'exp', '--ternary', '0', '--activation-aware', '0'],            # no quantization at all (distilled FP student)
    'ternary_expdecay': ['--decay', 'exp', '--ternary', '1'],                                   # ternary weights, unquantized decay
}
TOTAL_STEPS = int(TOKENS // (MICRO_BS * ACCUM * SEQ_LEN))
print('steps per run:', TOTAL_STEPS, '| runs planned:', len(SEEDS) * len(VARIANTS_TO_RUN))

In [ ]:
# ---------------- helpers: watchdog runner, hashing, restore ----------------
def sha256(p, n=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()

def kill_group(proc):
    for sig, wait in ((signal.SIGTERM, 15), (signal.SIGKILL, 5)):
        try:
            os.killpg(proc.pid, sig)
        except ProcessLookupError:
            return
        t = time.time()
        while proc.poll() is None and time.time() - t < wait:
            time.sleep(1)
        if proc.poll() is not None:
            return

SHOW = ('EVAL', 'steps x', 'done ->', 'Traceback', 'Error', 'all tests passed', '[ok]')
def run_cmd(cmd, log_path, ready_marker=None, ready_file=None, grace=HANG_GRACE_S, timeout_s=3 * 3600):
    # Runs cmd, logs everything, and kills it if it hangs after finishing (ready marker/file) or exceeds timeout.
    t0, ready_at, state = time.time(), [None], {'hung_killed': False, 'timed_out': False}
    with open(log_path, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n')
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                                cwd=PROJECT, start_new_session=True)
        def pump():
            for line in proc.stdout:
                lf.write(line); lf.flush()
                if ready_marker and ready_marker in line and ready_at[0] is None:
                    ready_at[0] = time.time()
                if any(s in line for s in SHOW):
                    print(line.rstrip()[:220])
        th = threading.Thread(target=pump, daemon=True); th.start()
        while proc.poll() is None:
            time.sleep(3)
            if ready_file and ready_at[0] is None and Path(ready_file).exists():
                ready_at[0] = time.time()
            if ready_at[0] and time.time() - ready_at[0] > grace:
                state['hung_killed'] = True; kill_group(proc); break
            if time.time() - t0 > timeout_s:
                state['timed_out'] = True; kill_group(proc); break
        th.join(timeout=10)
    return {'rc': proc.returncode, 'seconds': round(time.time() - t0, 1), **state}

def restore_previous():
    # Copies finished runs from a previous session's output (added as a Kaggle input) so they are skipped.
    n = 0
    for f in glob.glob('/kaggle/input/**/DONE.json', recursive=True):
        src = Path(f).parent
        dst = RESULTS / src.name
        if not dst.exists():
            shutil.copytree(src, dst); n += 1
    print(f'restored {n} finished runs from previous sessions')
restore_previous()

def env_info():
    smi = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    import transformers, datasets
    try:
        import lm_eval; lmv = getattr(lm_eval, '__version__', 'unknown')
    except Exception:
        lmv = 'missing'
    return {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
            'transformers': transformers.__version__, 'datasets': datasets.__version__, 'lm_eval': lmv,
            'gpu': smi, 'platform': platform.platform()}
ENV = env_info(); print(ENV)
SOURCE_HASHES = {p.name: sha256(p) for p in sorted(PROJECT.glob('*.py'))}

In [ ]:
# ---------------- 1) correctness tests (must pass before spending GPU hours) ----------------
r = run_cmd([sys.executable, '-u', 'tests.py'], RESULTS / 'tests.log', timeout_s=1800)
assert r['rc'] == 0 and 'all tests passed' in (RESULTS / 'tests.log').read_text(), 'tests.py failed, see results/tests.log'
print('tests passed')

In [ ]:
# ---------------- 2) teacher eval, once, same tasks and dtype as students ----------------
def eval_cmd(out_json, ckpt=None):
    c = [sys.executable, '-u', 'eval_lm.py', '--tasks', EVAL_TASKS, '--ppl-length', '--dtype', DTYPE, '--out', str(out_json)]
    return c + (['--ckpt', str(ckpt)] if ckpt else ['--pretrained', PRETRAINED])

T = RESULTS / 'teacher'; T.mkdir(exist_ok=True)
if not (T / 'DONE.json').exists():
    r = run_cmd(eval_cmd(T / 'eval.json'), T / 'eval.log', ready_file=T / 'eval.json', timeout_s=EVAL_TIMEOUT_S)
    assert (T / 'eval.json').exists(), 'teacher eval produced no JSON, see results/teacher/eval.log'
    json.dump({'run': 'teacher', 'eval_tasks': EVAL_TASKS, 'dtype': DTYPE, 'env': ENV, 'run_info': r,
               'finished_utc': datetime.datetime.utcnow().isoformat() + 'Z'}, open(T / 'DONE.json', 'w'), indent=1)
print('teacher ready')

In [ ]:
# ---------------- 3) the matrix: seed-major order so partial completion still gives paired data ----------------
def last_eval_record(log_jsonl):
    rec = None
    for line in open(log_jsonl):
        try:
            d = json.loads(line)
        except Exception:
            continue
        if 'student_ppl' in d:
            rec = d
    return rec

def avg_minutes(default=45.0):
    m = []
    for f in glob.glob(str(RESULTS / 'k_*' / 'DONE.json')):
        try: m.append(json.load(open(f))['minutes_total'])
        except Exception: pass
    return sum(m) / len(m) if m else default

def run_one(variant, seed):
    name = f'k_{variant}_s{seed}'
    out = RESULTS / name; out.mkdir(exist_ok=True)
    run_dir = RUNS / name
    ckpt = run_dir / 'last.pt'
    t_start, info = time.time(), {}
    train_cmd = [sys.executable, '-u', 'train_qat.py', '--name', name, '--seed', str(seed), '--out', str(RUNS)] + BASE + VARIANTS[variant]
    ok = False
    for attempt in (1, 2):
        cmd = train_cmd + (['--resume', str(ckpt)] if attempt == 2 and ckpt.exists() else [])
        info[f'train_attempt{attempt}'] = run_cmd(cmd, out / 'train.log', ready_marker='done ->', timeout_s=TRAIN_TIMEOUT_S)
        rec = last_eval_record(run_dir / 'log.jsonl') if (run_dir / 'log.jsonl').exists() else None
        if rec and rec['step'] == TOTAL_STEPS and ckpt.exists():
            ok = True; break
        print(f'[{name}] attempt {attempt} did not reach step {TOTAL_STEPS}')
    if not ok:
        json.dump({'run': name, 'status': 'train_failed', 'info': info}, open(out / 'FAILED.json', 'w'), indent=1)
        return False
    ev_json = out / 'eval.json'
    info['eval'] = run_cmd(eval_cmd(ev_json, ckpt), out / 'eval.log', ready_file=ev_json, timeout_s=EVAL_TIMEOUT_S)
    if not ev_json.exists():
        json.dump({'run': name, 'status': 'eval_failed', 'info': info}, open(out / 'FAILED.json', 'w'), indent=1)
        return False
    for f in ('log.jsonl', 'args.json', 'size_report.json', 'run_manifest.json'):
        if (run_dir / f).exists(): shutil.copy(run_dir / f, out / f)
    done = {'run': name, 'variant': variant, 'seed': seed, 'tokens': TOKENS, 'dtype': DTYPE, 'eval_tasks': EVAL_TASKS,
            'train_command': ' '.join(train_cmd), 'final_train_eval': rec, 'env': ENV, 'source_sha256': SOURCE_HASHES,
            'run_info': info, 'minutes_total': round((time.time() - t_start) / 60, 1),
            'artifact_sha256': {p.name: sha256(p) for p in sorted(out.iterdir()) if p.is_file()},
            'finished_utc': datetime.datetime.utcnow().isoformat() + 'Z'}
    json.dump(done, open(out / 'DONE.json', 'w'), indent=1)
    if not KEEP_CKPT:
        shutil.rmtree(run_dir, ignore_errors=True)
    shutil.make_archive(str(WORK / 'results_bundle'), 'zip', RESULTS)   # refreshed after every run
    return True

T_START, stop = time.time(), False
for seed in SEEDS:
    for v in VARIANTS_TO_RUN:
        name = f'k_{v}_s{seed}'
        if (RESULTS / name / 'DONE.json').exists():
            print('skip (done):', name); continue
        used_min = (time.time() - T_START) / 60
        if used_min + 1.15 * avg_minutes() > SESSION_BUDGET_H * 60:
            print(f'Stopping cleanly: ~{used_min:.0f} min used, next run would exceed the {SESSION_BUDGET_H} h budget.')
            print('Re-run with this notebook\'s output added as an input to continue.')
            stop = True; break
        print(f'\n=== {name} ===  ({used_min:.0f} min used)')
        try:
            run_one(v, seed)
        except Exception as e:
            print('run failed with exception:', repr(e))
    if stop: break
print('finished loop; completed runs:', len(glob.glob(str(RESULTS / 'k_*' / 'DONE.json'))))

In [ ]:
# ---------------- 4) aggregate: per-variant stats and PAIRED comparison against nearest ----------------
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

def metrics_from_eval(ej):
    r = ej.get('results', {})
    g = lambda t, k: r.get(t, {}).get(k)
    row = {'wikitext_ppl': g('wikitext', 'word_perplexity,none'), 'lambada_ppl': g('lambada_openai', 'perplexity,none'),
           'lambada_acc': g('lambada_openai', 'acc,none'), 'piqa_accnorm': g('piqa', 'acc_norm,none'),
           'arc_easy_accnorm': g('arc_easy', 'acc_norm,none'), 'arc_challenge_accnorm': g('arc_challenge', 'acc_norm,none'),
           'hellaswag_accnorm': g('hellaswag', 'acc_norm,none'), 'winogrande_acc': g('winogrande', 'acc,none')}
    lbp = ej.get('loss_by_position') or {}
    row['mean_loss_by_position'] = float(np.mean(list(lbp.values()))) if lbp else None
    return row

def load_runs(results_dir=RESULTS):
    rows = []
    for f in sorted(Path(results_dir).glob('k_*/DONE.json')):
        d = json.load(open(f)); ej = json.load(open(f.parent / 'eval.json'))
        rec = d.get('final_train_eval') or {}
        rows.append({'variant': d['variant'], 'seed': d['seed'], 'fineweb_val_ppl': rec.get('student_ppl'),
                     'k_mean': rec.get('k_mean'), 'frac_at_kmin': rec.get('frac_at_kmin'), **metrics_from_eval(ej)})
    return pd.DataFrame(rows)

LOWER_BETTER_LOG = ['wikitext_ppl', 'lambada_ppl', 'fineweb_val_ppl']          # compared as log-ratio (% change)
HIGHER_BETTER_ABS = ['lambada_acc', 'piqa_accnorm', 'arc_easy_accnorm', 'hellaswag_accnorm']  # compared as point difference

def paired_table(df, baseline='nearest'):
    out = []
    for var in sorted(set(df.variant) - {baseline}):
        for m in LOWER_BETTER_LOG + HIGHER_BETTER_ABS:
            a = df[df.variant == baseline].set_index('seed')[m].dropna()
            b = df[df.variant == var].set_index('seed')[m].dropna()
            seeds = sorted(set(a.index) & set(b.index)); n = len(seeds)
            if n < 2:
                out.append({'variant': var, 'metric': m, 'n_seeds': n, 'verdict': 'insufficient seeds'}); continue
            if m in LOWER_BETTER_LOG:
                d = np.log(b.loc[seeds].values / a.loc[seeds].values); scale = 'pct'
            else:
                d = (b.loc[seeds].values - a.loc[seeds].values) * 100; scale = 'pts'
            mean, sd = d.mean(), d.std(ddof=1)
            tcrit = stats.t.ppf(0.975, n - 1); half = tcrit * sd / math.sqrt(n)
            p = stats.ttest_1samp(d, 0).pvalue if sd > 0 else float('nan')
            conv = (lambda x: (math.exp(x) - 1) * 100) if scale == 'pct' else (lambda x: x)
            lo, hi = conv(mean - half), conv(mean + half)
            better = (mean < 0) if m in LOWER_BETTER_LOG else (mean > 0)
            verdict = 'no detectable difference' if lo <= 0 <= hi else ('better than baseline' if better else 'worse than baseline')
            out.append({'variant': var, 'metric': m, 'n_seeds': n, 'unit': '% change' if scale == 'pct' else 'points',
                        'mean_diff': conv(mean), 'ci95_lo': lo, 'ci95_hi': hi, 'p_value': p,
                        'seeds_better': int((d < 0).sum() if m in LOWER_BETTER_LOG else (d > 0).sum()),
                        'min_detectable_effect': conv(half), 'verdict': verdict})
    return pd.DataFrame(out)

def summarize(df):
    cols = LOWER_BETTER_LOG + HIGHER_BETTER_ABS + ['arc_challenge_accnorm', 'winogrande_acc', 'k_mean', 'frac_at_kmin']
    g = df.groupby('variant')[cols].agg(['mean', 'std', 'count'])
    return g

def paired_plot(df, path, metric='wikitext_ppl'):
    order = [v for v in ['fp_student', 'nearest', 'ternary_expdecay', 'harpq', 'dyadic2'] if v in set(df.variant)]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    pv = df.pivot_table(index='seed', columns='variant', values=metric)
    for s, row in pv.iterrows():
        xs = [i for i, v in enumerate(order) if not np.isnan(row.get(v, np.nan))]
        ax.plot(xs, [row[order[i]] for i in xs], '-o', alpha=.55, lw=1, ms=4, label=f'seed {s}')
    means = [pv[v].mean() for v in order]
    ax.plot(range(len(order)), means, 'k_', ms=22, mew=2, label='mean')
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order); ax.set_ylabel(metric); ax.set_yscale('log')
    ax.set_title('Same seed = same line (paired)'); ax.legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)

def build_report(results_dir=RESULTS, baseline='nearest'):
    results_dir = Path(results_dir); df = load_runs(results_dir)
    if df.empty: print('no finished runs yet'); return None
    df.to_csv(results_dir / 'per_run.csv', index=False)
    summ = summarize(df); summ.to_csv(results_dir / 'summary_mean_std.csv')
    pt = paired_table(df, baseline); pt.to_csv(results_dir / 'paired_vs_nearest.csv', index=False)
    paired_plot(df, results_dir / 'paired_wikitext.png')
    teacher = json.load(open(results_dir / 'teacher' / 'eval.json')) if (results_dir / 'teacher' / 'eval.json').exists() else None
    lines = ['# Results (auto-generated; every number comes from executed runs)', '',
             f'Runs: {len(df)} | variants: {sorted(df.variant.unique())} | seeds per variant: ' +
             str(df.groupby("variant").seed.nunique().to_dict()), '']
    if teacher: lines += ['Teacher: ' + json.dumps({k: (round(v, 4) if v else v) for k, v in metrics_from_eval(teacher).items()}), '']
    for _, r in pt[pt.metric.isin(['wikitext_ppl', 'lambada_acc'])].iterrows():
        if r['verdict'] == 'insufficient seeds':
            lines.append(f"- {r['variant']} / {r['metric']}: insufficient seeds (n={r['n_seeds']})")
        else:
            lines.append(f"- {r['variant']} vs {baseline} / {r['metric']}: {r['mean_diff']:+.2f} {r['unit']} "
                         f"(95% CI {r['ci95_lo']:+.2f} to {r['ci95_hi']:+.2f}, n={r['n_seeds']}, p={r['p_value']:.3f}, "
                         f"better in {int(r['seeds_better'])}/{int(r['n_seeds'])} seeds; smallest effect this design can detect: "
                         f"+/-{r['min_detectable_effect']:.2f}) -> {r['verdict']}")
    (results_dir / 'REPORT.md').write_text('\n'.join(lines))
    print('\n'.join(lines))
    return df, summ, pt

out = build_report()
if out:
    pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
    display(out[1].round(4)); display(out[2].round(4))

## How to read the output and what you may claim

- `results/REPORT.md`, `per_run.csv`, `summary_mean_std.csv`, `paired_vs_nearest.csv`, `paired_wikitext.png`, plus `results_bundle.zip` with everything.
- A difference counts only when the 95% CI excludes 0. The report also prints the smallest effect your seed count can detect; if that is larger than the effect you care about, run more seeds before claiming anything.
- Compare only runs from this notebook with each other. Do **not** pool them with the earlier Colab/Kaggle pilots or the `fast2M_s2` run.
- `fp_student` is an untested path in your code (`--decay exp --ternary 0`); if it errors, the run is logged as FAILED and the loop continues.
- No speed or energy claim is supported: the kernel microbenchmarks showed the ternary Triton kernel slower than fp16.
- State the scale honestly: Mamba2-130M, 2M training tokens per run.